In [ ]:
%pip install pandas azure-storage-blob
import sys
from pathlib import Path
import os

ROOT_DIR = Path("c:\\Users\\rafae\\OneDrive\\Documentos\\F1")
sys.path.append(str(ROOT_DIR / "pipeline"))
import pandas as pd
from utils.storage import AzureBlobStorage
import logging
import json

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

AZURE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
RAW_CONTAINER = "raw"
SILVER_CONTAINER = "processed"

storage_raw = AzureBlobStorage(AZURE_CONNECTION_STRING, RAW_CONTAINER)
storage_silver = AzureBlobStorage(AZURE_CONNECTION_STRING, SILVER_CONTAINER)


def normalize_columns(df):
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df


def load_json_file(filename):
    full_path = filename if filename.startswith("raw/") else f"raw/{filename}"
    
    logging.info(f"Lendo blob: {full_path}")
    data = storage_raw.download_json(full_path)
    df = pd.DataFrame(data if isinstance(data, list) else [data])
    df = normalize_columns(df)
    return df

df_position = load_json_file("position.json")
logging.info(f"position.json carregado: {len(df_position)} linhas")

df_race = load_json_file("race.json")
logging.info(f"race.json carregado: {len(df_race)} linhas")

df_race_result = load_json_file("race_result.json")
logging.info(f"race_result.json carregado: {len(df_race_result)} linhas")

df_drivers = load_json_file("drivers.json")
logging.info(f"drivers.json carregado: {len(df_drivers)} linhas")

df_race_control = load_json_file("race_control.json")
logging.info(f"race_control.json carregado: {len(df_race_control)} linhas")

df_weather = load_json_file("weather.json")
logging.info(f"weather.json carregado: {len(df_weather)} linhas")

logging.info("Todos os arquivos foram carregados")


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


2025-12-06 02:11:46,489 - INFO - === Carregando arquivos JSON ===
2025-12-06 02:11:46,490 - INFO - Lendo blob: raw/position.json
2025-12-06 02:11:46,490 - INFO - Request URL: 'https://stf1analytics01.blob.core.windows.net/raw/raw/position.json'
Request method: 'GET'
Request headers:
    'x-ms-range': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.27.1 Python/3.11.0 (Windows-10-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '0ff62ff8-d262-11f0-96db-a71f4788a230'
    'Authorization': 'REDACTED'
No body was attached to the request
2025-12-06 02:11:46,583 - INFO - Response status: 206
Response headers:
    'Content-Length': '3261320'
    'Content-Type': 'application/octet-stream'
    'Content-Range': 'REDACTED'
    'Last-Modified': 'Sat, 06 Dec 2025 03:36:29 GMT'
    'Accept-Ranges': 'REDACTED'
    'ETag': '"0x8DE3478A4C0E298"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'


In [2]:
df_drivers = df_drivers[:20]
df_drivers = df_drivers.drop(columns=["meeting_key", "session_key", "broadcast_name", "team_colour", "first_name", "last_name", "headshot_url"])
df_drivers

,driver_number,full_name,name_acronym,team_name,country_code
0,1,Max VERSTAPPEN,VER,Red Bull Racing,NED
1,2,Logan SARGEANT,SAR,Williams,USA
2,4,Lando NORRIS,NOR,McLaren,GBR
3,10,Pierre GASLY,GAS,Alpine,FRA
4,11,Sergio PEREZ,PER,Red Bull Racing,MEX
5,14,Fernando ALONSO,ALO,Aston Martin,ESP
6,16,Charles LECLERC,LEC,Ferrari,MON
7,18,Lance STROLL,STR,Aston Martin,CAN
8,20,Kevin MAGNUSSEN,MAG,Haas F1 Team,DEN
9,21,Nyck DE VRIES,DEV,AlphaTauri,NED


In [3]:
df_race.head()
df_race = df_race.drop(columns=["meeting_key", "session_type", "session_name", "country_key", "circuit_key", "gmt_offset", "year","date_end", "date_start"])
df_race.shape
df_race.head(23)

,session_key,location,country_code,country_name,circuit_short_name
0,9472,Sakhir,BRN,Bahrain,Sakhir
1,9480,Jeddah,KSA,Saudi Arabia,Jeddah
2,9488,Melbourne,AUS,Australia,Melbourne
3,9496,Suzuka,JPN,Japan,Suzuka
4,9673,Shanghai,CHN,China,Shanghai
5,9507,Miami,USA,United States,Miami
6,9515,Imola,ITA,Italy,Imola
7,9523,Monaco,MON,Monaco,Monte Carlo
8,9531,Montréal,CAN,Canada,Montreal
9,9539,Barcelona,ESP,Spain,Catalunya


In [4]:
df_race_control.head()
df_race_control = df_race_control.drop(columns=["meeting_key", "scope", "sector", "date", 'message'])
df_race_control

,session_key,driver_number,lap_number,category,flag
0,9475,1,NaN,CarEvent,None
1,9480,4,41.0,Flag,BLACK AND WHITE
2,9515,1,24.0,Flag,BLACK AND WHITE
3,9550,4,57.0,Flag,BLACK AND WHITE
4,9577,1,NaN,Flag,BLACK AND WHITE
5,9607,81,NaN,Flag,BLACK AND WHITE
6,9617,81,46.0,Flag,BLACK AND WHITE
7,9617,4,56.0,Flag,BLACK AND WHITE
8,9693,81,46.0,Flag,BLUE
9,9693,81,47.0,Flag,BLUE


In [5]:
df_race_control_agg = (
    df_race_control
        .sort_values(["session_key", "driver_number", "lap_number"])
        .groupby(["session_key", "driver_number"], as_index=False)
        .first()
)
df_race_control_agg


,session_key,driver_number,lap_number,category,flag
0,9475,1,NaN,CarEvent,None
1,9480,4,41.0,Flag,BLACK AND WHITE
2,9515,1,24.0,Flag,BLACK AND WHITE
3,9550,4,57.0,Flag,BLACK AND WHITE
4,9577,1,NaN,Flag,BLACK AND WHITE
5,9607,81,NaN,Flag,BLACK AND WHITE
6,9617,4,56.0,Flag,BLACK AND WHITE
7,9617,81,46.0,Flag,BLACK AND WHITE
8,9693,81,46.0,Flag,BLUE
9,9888,4,21.0,Flag,BLACK AND WHITE


In [6]:
valid_keys = df_race["session_key"].unique()
df_race_result_filtered = df_race_result[df_race_result["session_key"].isin(valid_keys)]
df_race_result_filtered = df_race_result_filtered.drop(columns=["meeting_key"])
df_race_result_filtered.shape
df_race_result_filtered.head(69)


,position,driver_number,number_of_laps,dnf,dns,dsq,gap_to_leader,duration,session_key,points
345,1.0,1,57.0,False,False,False,0,5504.742,9472,26.0
346,6.0,4,57.0,False,False,False,48.458,5553.2,9472,8.0
347,8.0,81,57.0,False,False,False,56.082,5560.824,9472,4.0
360,1.0,1,50.0,False,False,False,0,4843.273,9480,25.0
361,4.0,81,50.0,False,False,False,32.007,4875.28,9480,12.0
...,...,...,...,...,...,...,...,...,...,...
659,6.0,4,50.0,False,False,False,43.385,4969.354,9644,9.0
660,7.0,81,50.0,False,False,False,51.365,4977.334,9644,6.0
673,1.0,1,57.0,False,False,False,0,5465.323,9655,25.0
674,3.0,81,57.0,False,False,False,6.819,5472.142,9655,15.0


In [7]:
df_position.shape

(26365, 5)

In [8]:

df_position_filtered = df_position[df_position["session_key"].isin(valid_keys)]
df_position_filtered["date"] = pd.to_datetime(df_position_filtered["date"], format="mixed")


df_position_filtered["day"] = df_position_filtered["date"].dt.date

df_first_position = (
    df_position_filtered
        .sort_values(["day", "date"])
        .drop_duplicates(subset=["day", "session_key", "driver_number"], keep="first")
)

df_first_position = df_first_position.drop(columns=["meeting_key", "date"])

df_first_position.head(50)



C:\Users\rafae\AppData\Local\Temp\ipykernel_18792\2371582593.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_position_filtered["date"] = pd.to_datetime(df_position_filtered["date"], format="mixed")
C:\Users\rafae\AppData\Local\Temp\ipykernel_18792\2371582593.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_position_filtered["day"] = df_position_filtered["date"].dt.date


,session_key,driver_number,position,day
713,9472,1,1,2024-03-02
714,9472,4,7,2024-03-02
715,9472,81,8,2024-03-02
1322,9480,1,1,2024-03-09
1323,9480,81,5,2024-03-09
1324,9480,4,6,2024-03-09
1771,9488,1,1,2024-03-24
1772,9488,4,3,2024-03-24
1773,9488,81,5,2024-03-24
2090,9496,1,1,2024-04-07


In [9]:
df_first_position

,session_key,driver_number,position,day
713,9472,1,1,2024-03-02
714,9472,4,7,2024-03-02
715,9472,81,8,2024-03-02
1322,9480,1,1,2024-03-09
1323,9480,81,5,2024-03-09
...,...,...,...,...
24969,9858,1,2,2025-11-23
24970,9858,81,5,2025-11-23
25649,9850,81,1,2025-11-30
25650,9850,4,2,2025-11-30


In [10]:
df_weather

,date,session_key,humidity,pressure,rainfall,track_temperature,wind_speed,meeting_key,wind_direction,air_temperature
0,2024-02-21T06:55:35.254000+00:00,9462,61.0,1020.5,0,28.6,0.9,1228,109,21.8
1,2024-02-21T06:56:35.271000+00:00,9462,61.0,1020.5,0,28.6,1.1,1228,1,21.8
2,2024-02-21T06:57:35.289000+00:00,9462,60.0,1020.5,0,28.9,0.7,1228,48,21.9
3,2024-02-21T06:58:35.293000+00:00,9462,59.0,1020.4,0,29.2,1.1,1228,9,22.0
4,2024-02-21T06:59:35.279000+00:00,9462,58.0,1020.4,0,29.4,1.1,1228,55,22.1
...,...,...,...,...,...,...,...,...,...,...
26191,2025-07-06T15:38:00.887000+00:00,9947,65.0,989.1,0,24.4,1.1,1277,95,19.3
26192,2025-07-06T15:39:00.899000+00:00,9947,64.0,989.0,0,24.4,1.9,1277,230,19.4
26193,2025-07-06T15:40:00.890000+00:00,9947,65.0,989.0,0,24.6,1.5,1277,287,19.5
26194,2025-07-06T15:41:00.890000+00:00,9947,64.0,989.0,0,24.6,1.6,1277,48,19.5


In [11]:
df_weather.describe()

,session_key,humidity,pressure,rainfall,track_temperature,wind_speed,meeting_key,wind_direction,air_temperature
count,26196.000000,26196.000000,26196.000000,26196.000000,26196.000000,26196.000000,26196.000000,26196.000000,26196.000000
mean,9726.958620,52.379409,991.879818,0.048175,34.581108,1.897885,1251.703199,165.413307,22.989399
std,190.790807,16.342269,47.491249,0.214141,9.391928,1.120535,14.415397,104.466396,5.189250
min,9461.000000,11.000000,780.300000,0.000000,12.000000,0.000000,1228.000000,0.000000,10.700000
25%,9558.000000,39.000000,992.000000,0.000000,27.800000,1.100000,1239.000000,65.000000,19.200000
50%,9683.000000,52.000000,1008.600000,0.000000,35.500000,1.700000,1253.000000,172.000000,23.000000
75%,9915.000000,64.000000,1016.900000,0.000000,41.800000,2.500000,1264.000000,245.000000,27.100000
max,10033.000000,96.000000,1025.600000,1.000000,60.600000,9.600000,1277.000000,360.000000,35.500000


In [12]:
df_weather = df_weather.drop(columns=["date"])

In [13]:
df_weather_agg = (
    df_weather.groupby("session_key")
        .agg(
            wind_speed_mean=("wind_speed", "mean"),

            wind_direction_mean=("wind_direction", "mean"),

            rainfall_mean=("rainfall", "mean"),

            track_temperature_mean=("track_temperature", "mean"),

            air_temperature_mean=("air_temperature", "mean"),

            humidity_mean=("humidity", "mean"),

            pressure_mean=("pressure", "mean")
        )
        .reset_index()
)


In [14]:
df_weather_agg_filterad = df_weather_agg[df_weather_agg["session_key"].isin(valid_keys)]
df_weather_agg_filterad.shape

(47, 8)

In [15]:
df_weather_agg_filterad

,session_key,wind_speed_mean,wind_direction_mean,rainfall_mean,track_temperature_mean,air_temperature_mean,humidity_mean,pressure_mean
8,9472,0.785987,89.796178,0.000000,23.652866,18.227389,48.821656,1017.185987
13,9480,1.302740,129.143836,0.000000,31.593151,25.528082,62.335616,1012.682877
18,9488,0.953472,154.784722,0.000000,38.402083,20.622222,44.451389,1020.601389
23,9496,2.240884,185.668508,0.000000,37.501105,21.691160,43.430939,1012.372376
28,9507,3.080667,152.640000,0.000000,44.664000,28.522000,59.006667,1016.462667
33,9515,2.141259,135.979021,0.000000,42.023776,25.113986,48.426573,1006.634266
38,9523,0.979500,160.895000,0.000000,46.318000,21.596500,63.990000,1018.743500
43,9531,1.169912,158.150442,0.274336,24.569912,17.797345,73.380531,996.197345
48,9539,2.123377,207.370130,0.006494,41.096104,24.132468,63.441558,1001.356494
53,9550,1.667832,235.510490,0.006993,46.420979,29.425874,35.006993,932.862937


In [16]:
df = (
    df_race_result_filtered
        .merge(df_race, on="session_key", how="left")
        .merge(df_race_control_agg , on=["session_key", "driver_number"], how="left")
        .merge(df_first_position, on=["session_key", "driver_number"], how="left")
        .merge(df_weather_agg_filterad, on="session_key", how="left")
        .merge(df_drivers, on="driver_number", how="left")
)
df


,position_x,driver_number,number_of_laps,dnf,dns,dsq,gap_to_leader,duration,session_key,points,...,wind_direction_mean,rainfall_mean,track_temperature_mean,air_temperature_mean,humidity_mean,pressure_mean,full_name,name_acronym,team_name,country_code_y
0,1.0,1,57.0,False,False,False,0,5504.742,9472,26.0,...,89.796178,0.000000,23.652866,18.227389,48.821656,1017.185987,Max VERSTAPPEN,VER,Red Bull Racing,NED
1,6.0,4,57.0,False,False,False,48.458,5553.2,9472,8.0,...,89.796178,0.000000,23.652866,18.227389,48.821656,1017.185987,Lando NORRIS,NOR,McLaren,GBR
2,8.0,81,57.0,False,False,False,56.082,5560.824,9472,4.0,...,89.796178,0.000000,23.652866,18.227389,48.821656,1017.185987,Oscar PIASTRI,PIA,McLaren,AUS
3,1.0,1,50.0,False,False,False,0,4843.273,9480,25.0,...,129.143836,0.000000,31.593151,25.528082,62.335616,1012.682877,Max VERSTAPPEN,VER,Red Bull Racing,NED
4,4.0,81,50.0,False,False,False,32.007,4875.28,9480,12.0,...,129.143836,0.000000,31.593151,25.528082,62.335616,1012.682877,Oscar PIASTRI,PIA,McLaren,AUS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,2.0,81,57.0,False,False,False,7.995,5086.236,9850,18.0,...,216.463087,0.000000,28.160403,22.599329,69.046980,1016.095302,Oscar PIASTRI,PIA,McLaren,AUS
137,4.0,4,57.0,False,False,False,23.315,5101.556,9850,12.0,...,216.463087,0.000000,28.160403,22.599329,69.046980,1016.095302,Lando NORRIS,NOR,McLaren,GBR
138,1.0,4,52.0,False,False,False,0,5835.735,9947,25.0,...,139.006452,0.180645,23.578710,17.917419,78.374194,988.776774,Lando NORRIS,NOR,McLaren,GBR
139,2.0,81,52.0,False,False,False,6.812,5842.547,9947,18.0,...,139.006452,0.180645,23.578710,17.917419,78.374194,988.776774,Oscar PIASTRI,PIA,McLaren,AUS


In [17]:
df = df.drop(columns=["position_y"])
df = df.rename(columns={"position_x": "position"})

In [18]:
df.groupby("session_key")["driver_number"].nunique().reset_index(name="qtd_pilotos")

,session_key,qtd_pilotos
0,9472,3
1,9480,3
2,9488,3
3,9496,3
4,9507,3
5,9515,3
6,9523,3
7,9531,3
8,9539,3
9,9550,3


In [19]:
df["year"] = df["day"].apply(lambda x: x.year)

In [20]:
def flatten_list_columns(df):
    for col in df.columns:
        if df[col].dtype == object:
            has_list = any(isinstance(val, list) for val in df[col].dropna())
            has_mixed = False
            
            if not has_list:
                types_set = set()
                for val in df[col].dropna():
                    types_set.add(type(val).__name__)
                has_mixed = len(types_set) > 1
            
            if has_list or has_mixed:
                logging.info(f"Convertendo coluna '{col}' (contém listas ou tipos mistos) para string")
                df[col] = df[col].apply(
                    lambda x: json.dumps(x) if isinstance(x, (list, dict)) else str(x) if pd.notna(x) else None
                )
                df[col] = df[col].astype(str)
    
    return df

In [ ]:
def save_silver(df, filename="silver_data.parquet"):
    logging.info(f"Processando datatyoes antes de salvar")

    df = flatten_list_columns(df)
    df.to_parquet(filename, engine='pyarrow', index=False)
    
    logging.info(f"Fazendo upload para {SILVER_CONTAINER}/{filename}")
    with open(filename, "rb") as f:
        storage_silver.container_client.upload_blob(name=filename, data=f, overwrite=True)
    
    logging.info(f"Upload concluido ")

save_silver(df, filename="silver_data.parquet")

2025-12-06 02:11:47,382 - INFO - Processando DataTypes antes de salvar...
2025-12-06 02:11:47,383 - INFO - Convertendo coluna 'gap_to_leader' (contém listas ou tipos mistos) para string
2025-12-06 02:11:47,406 - INFO - Fazendo upload para processed/silver_data.parquet
2025-12-06 02:11:47,416 - INFO - Request URL: 'https://stf1analytics01.blob.core.windows.net/processed/silver_data.parquet'
Request method: 'PUT'
Request headers:
    'Content-Length': '25321'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.27.1 Python/3.11.0 (Windows-10-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '1083676f-d262-11f0-9e13-a71f4788a230'
    'Authorization': 'REDACTED'
A body is sent with the request
2025-12-06 02:11:47,492 - INFO - Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-